#Milestone 2 — Derive and Implement a Gradient
##Implement forward and backward passes in NumPy;

In [8]:
import numpy as np


def attention_forward(Q, K, V):
  d_k = Q.shape[-1]
  S = np.matmul(Q, K.T) / np.sqrt(d_k)
  # Numerical stability fix: shift by max
  S_max = S - np.max(S, axis=-1, keepdims=True)
  exp_S = np.exp(S_max)
  P = exp_S / np.sum(exp_S, axis=-1, keepdims=True)
  O = np.matmul(P, V)
  return O, P, S


def attention_backward(Q, K, V, P, dO):
  d_k = Q.shape[-1]
  dV = np.matmul(P.T, dO)
  dP = np.matmul(dO, V.T)

  # Softmax backward
  sum_dp_p = np.sum(dP * P, axis=-1, keepdims=True)
  dS = P * (dP - sum_dp_p)

  # Scale factor and linear projections backward
  dQ = np.matmul(dS, K) / np.sqrt(d_k)
  dK = np.matmul(dS.T, Q) / np.sqrt(d_k)
  return dQ, dK, dV

##Example Execution

In [9]:
Q = np.array([[1.0, 0.0, 1.0], [0.5, 1.0, 0.0], [1.0, 1.0, 1.0], [0.0, 0.5, 0.5]])
K = Q.copy()
V = np.array([[1.0, 0.5], [0.0, 1.0], [1.0, 1.0], [0.5, 0.0]])
dO = np.ones((4, 2))  # Dummy upstream gradient

# 1. Run Forward
O, P, S = attention_forward(Q, K, V)
print("--- Forward Pass Outputs ---")
print("O shape:", O.shape)
print("P shape:", P.shape)

# 2. Run Backward
dQ, dK, dV = attention_backward(Q, K, V, P, dO)
print("\n--- Backward Pass Outputs ---")
print("dQ shape:", dQ.shape)
print("dK shape:", dK.shape)
print("dV shape:", dV.shape)

--- Forward Pass Outputs ---
O shape: (4, 2)
P shape: (4, 4)

--- Backward Pass Outputs ---
dQ shape: (4, 3)
dK shape: (4, 3)
dV shape: (4, 2)


In [11]:
import numpy as np


def stable_softmax(x, axis=-1):
  """Numerically stable softmax using max subtraction."""
  x = np.asarray(x, dtype=np.float64)
  shifted = x - np.max(x, axis=axis, keepdims=True)
  exps = np.exp(shifted)
  return exps / np.sum(exps, axis=axis, keepdims=True)


def attention_forward(Q, K, V, causal=False):
  """Return O and a cache for backward. Inputs are converted to float64."""
  Q = np.asarray(Q, dtype=np.float64)
  K = np.asarray(K, dtype=np.float64)
  V = np.asarray(V, dtype=np.float64)

  d = Q.shape[-1]
  scores = (Q @ K.T) / np.sqrt(d)

  mask = None
  if causal:
    if Q.shape[0] != K.shape[0]:
      raise ValueError("This simple causal mask expects equal query/key lengths.")
    mask = np.triu(np.ones(scores.shape, dtype=bool), k=1)
    scores = np.where(mask, -np.inf, scores)

  P = stable_softmax(scores, axis=-1)
  O = P @ V
  cache = (Q, K, V, P, mask)
  return O, cache


def attention_backward(dO, cache):
  """Compute analytical gradients dQ, dK, dV for the attention operation."""
  Q, K, V, P, mask = cache
  d = Q.shape[-1]

  dV = P.T @ dO
  dP = dO @ V.T

  # Vector-Jacobian product for row-wise softmax.
  dS = P * (dP - np.sum(dP * P, axis=-1, keepdims=True))

  # Masked score entries are constants (-inf), so their gradients are zero.
  if mask is not None:
    dS = np.where(mask, 0.0, dS)

  dQ = (dS @ K) / np.sqrt(d)
  dK = (dS.T @ Q) / np.sqrt(d)
  return dQ, dK, dV


# --- Gradient Verification Script ---
rng = np.random.default_rng(7)
Q = rng.normal(size=(4, 3))
K = rng.normal(size=(4, 3))
V = rng.normal(size=(4, 2))
O, _ = attention_forward(Q, K, V, causal=False)
dO = rng.normal(size=O.shape)

eps = 1e-5
threshold = 1e-6


def check_grad(param_name, analytic_grad, param_val, compute_loss_fn):
  numeric_grad = np.zeros_like(param_val)
  it = np.nditer(param_val, flags=["multi_index"], op_flags=["readwrite"])

  while not it.finished:
    idx = it.multi_index
    orig_val = param_val[idx]

    param_val[idx] = orig_val + eps
    loss_high = compute_loss_fn()

    param_val[idx] = orig_val - eps
    loss_low = compute_loss_fn()

    numeric_grad[idx] = (loss_high - loss_low) / (2 * eps)
    param_val[idx] = orig_val
    it.iternext()

  diff = np.linalg.norm(analytic_grad - numeric_grad)
  norm_sum = np.linalg.norm(analytic_grad) + np.linalg.norm(numeric_grad)
  rel_error = diff / norm_sum if norm_sum > 0 else 0.0

  print(
      f"[{param_name}] Relative Error: {rel_error:.2e} | Passed:"
      f" {rel_error < threshold}"
  )


def get_loss(q, k, v, causal_flag):
  o_out, _ = attention_forward(q, k, v, causal=causal_flag)
  return np.sum(o_out * dO)


# Run checks for both non-causal and causal attention
for causal_flag in [False, True]:
  print(f"\n--- Testing Gradient Check (causal={causal_flag}) ---")
  O_check, cache_check = attention_forward(Q, K, V, causal=causal_flag)
  dQ_an, dK_an, dV_an = attention_backward(dO, cache_check)

  check_grad("dQ", dQ_an, Q, lambda: get_loss(Q, K, V, causal_flag))
  check_grad("dK", dK_an, K, lambda: get_loss(Q, K, V, causal_flag))
  check_grad("dV", dV_an, V, lambda: get_loss(Q, K, V, causal_flag))


--- Testing Gradient Check (causal=False) ---
[dQ] Relative Error: 1.67e-11 | Passed: True
[dK] Relative Error: 4.03e-11 | Passed: True
[dV] Relative Error: 1.53e-11 | Passed: True

--- Testing Gradient Check (causal=True) ---
[dQ] Relative Error: 2.70e-11 | Passed: True
[dK] Relative Error: 1.70e-11 | Passed: True
[dV] Relative Error: 3.48e-12 | Passed: True
